In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    mutual_info_classif,
    chi2,
    RFE
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, Lasso

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
data = {
    "Age": [22, 25, 30, 35, 40, 45, 50, 55, 28, 38],
    "Income": [20000, 25000, 30000, 40000, 50000,
               60000, 70000, 80000, 28000, 45000],
    "Experience": [1, 2, 5, 8, 10, 15, 20, 25, 4, 9],
    "Credit_Score": [500, 520, 550, 600, 650,
                     700, 750, 800, 530, 620],
    "Random_Feature": [10, 50, 20, 80, 30, 90, 40, 70, 60, 15],
    "Target": [0, 0, 0, 0, 1, 1, 1, 1, 0, 1]
}

df = pd.DataFrame(data)

print("Original Dataset:")
display(df)

Original Dataset:


,Age,Income,Experience,Credit_Score,Random_Feature,Target
0,22,20000,1,500,10,0
1,25,25000,2,520,50,0
2,30,30000,5,550,20,0
3,35,40000,8,600,80,0
4,40,50000,10,650,30,1
5,45,60000,15,700,90,1
6,50,70000,20,750,40,1
7,55,80000,25,800,70,1
8,28,28000,4,530,60,0
9,38,45000,9,620,15,1


In [3]:
X = df.drop("Target", axis=1)
y = df["Target"]

print("Features:")
display(X)

print("Target:")
display(y)

Features:


,Age,Income,Experience,Credit_Score,Random_Feature
0,22,20000,1,500,10
1,25,25000,2,520,50
2,30,30000,5,550,20
3,35,40000,8,600,80
4,40,50000,10,650,30
5,45,60000,15,700,90
6,50,70000,20,750,40
7,55,80000,25,800,70
8,28,28000,4,530,60
9,38,45000,9,620,15


Target:


0    0
1    0
2    0
3    0
4    1
5    1
6    1
7    1
8    0
9    1
Name: Target, dtype: int64

In [4]:
selected_features = ["Age", "Income", "Credit_Score"]

X_selected = X[selected_features]

print("Selected Features:")
display(X_selected)

Selected Features:


,Age,Income,Credit_Score
0,22,20000,500
1,25,25000,520
2,30,30000,550
3,35,40000,600
4,40,50000,650
5,45,60000,700
6,50,70000,750
7,55,80000,800
8,28,28000,530
9,38,45000,620


In [5]:
print("Original Number of Features:", X.shape[1])
print("Selected Number of Features:", X_selected.shape[1])

Original Number of Features: 5
Selected Number of Features: 3


In [6]:
correlation = df.corr(numeric_only=True)

print("Feature Correlation with Target:")
display(correlation["Target"].drop("Target").sort_values(
    ascending=False
).to_frame("Correlation"))

Feature Correlation with Target:


,Correlation
Age,0.850887
Income,0.846371
Credit_Score,0.844151
Experience,0.787788
Random_Feature,0.093807


In [7]:
print("Random Feature Values:")
display(df[["Random_Feature", "Target"]])

print(
    "Random Feature Correlation:",
    round(df["Random_Feature"].corr(df["Target"]), 3)
)

Random Feature Values:


,Random_Feature,Target
0,10,0
1,50,0
2,20,0
3,80,0
4,30,1
5,90,1
6,40,1
7,70,1
8,60,0
9,15,1


Random Feature Correlation: 0.094


In [8]:
df["Income_Duplicate"] = df["Income"]

feature_correlation = df.drop(
    columns=["Target"]
).corr(numeric_only=True)

print("Feature Correlation Matrix:")
display(feature_correlation)

Feature Correlation Matrix:


,Age,Income,Experience,Credit_Score,Random_Feature,Income_Duplicate
Age,1.000000,0.996493,0.984821,0.995793,0.401998,0.996493
Income,0.996493,1.000000,0.991840,0.999519,0.406387,1.000000
Experience,0.984821,0.991840,1.000000,0.991332,0.411585,0.991840
Credit_Score,0.995793,0.999519,0.991332,1.000000,0.400571,0.999519
Random_Feature,0.401998,0.406387,0.411585,0.400571,1.000000,0.406387
Income_Duplicate,0.996493,1.000000,0.991840,0.999519,0.406387,1.000000


In [9]:
from sklearn.feature_selection import SelectKBest, f_classif

filter_selector = SelectKBest(
    score_func=f_classif,
    k=3
)

X_filter = filter_selector.fit_transform(X, y)

selected_filter_features = X.columns[
    filter_selector.get_support()
]

print("Filter Method Selected Features:")
print(selected_filter_features.tolist())

Filter Method Selected Features:
['Age', 'Income', 'Credit_Score']


In [10]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=2000)

wrapper_selector = RFE(
    estimator=model,
    n_features_to_select=3
)

wrapper_selector.fit(X, y)

selected_wrapper_features = X.columns[
    wrapper_selector.support_
]

print("Wrapper Method Selected Features:")
print(selected_wrapper_features.tolist())

Wrapper Method Selected Features:
['Age', 'Credit_Score', 'Random_Feature']


In [11]:
from sklearn.ensemble import RandomForestClassifier

embedded_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

embedded_model.fit(X, y)

importance = pd.Series(
    embedded_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("Embedded Feature Importance:")
display(importance.to_frame("Importance"))

Embedded Feature Importance:


,Importance
Credit_Score,0.27
Income,0.25
Age,0.24
Experience,0.23
Random_Feature,0.01


In [12]:
from sklearn.ensemble import RandomForestClassifier

embedded_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

embedded_model.fit(X, y)

importance = pd.Series(
    embedded_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("Embedded Feature Importance:")
display(importance.to_frame("Importance"))

Embedded Feature Importance:


,Importance
Credit_Score,0.27
Income,0.25
Age,0.24
Experience,0.23
Random_Feature,0.01


In [13]:
from sklearn.feature_selection import VarianceThreshold

variance_selector = VarianceThreshold(threshold=0.01)

X_variance = variance_selector.fit_transform(X)

selected_variance_features = X.columns[
    variance_selector.get_support()
]

print("Features After Variance Threshold:")
print(selected_variance_features.tolist())

Features After Variance Threshold:
['Age', 'Income', 'Experience', 'Credit_Score', 'Random_Feature']


In [14]:
correlation_matrix = X.corr().abs()

upper_triangle = correlation_matrix.where(
    np.triu(
        np.ones(correlation_matrix.shape),
        k=1
    ).astype(bool)
)

correlation_threshold = 0.90

high_correlation_features = [
    column
    for column in upper_triangle.columns
    if any(upper_triangle[column] > correlation_threshold)
]

print("Highly Correlated Features:")
print(high_correlation_features)

X_correlation = X.drop(
    columns=high_correlation_features
)

print("\nRemaining Features:")
print(X_correlation.columns.tolist())

Highly Correlated Features:
['Income', 'Experience', 'Credit_Score']

Remaining Features:
['Age', 'Random_Feature']


In [15]:
from sklearn.feature_selection import mutual_info_classif

mi_scores = mutual_info_classif(
    X,
    y,
    random_state=42
)

mi_results = pd.DataFrame({
    "Feature": X.columns,
    "Mutual_Information": mi_scores
}).sort_values(
    by="Mutual_Information",
    ascending=False
)

print("Mutual Information Scores:")
display(mi_results)

Mutual Information Scores:


,Feature,Mutual_Information
0,Age,0.383968
2,Experience,0.382063
1,Income,0.366349
3,Credit_Score,0.366349
4,Random_Feature,0.000000


In [16]:
from sklearn.feature_selection import chi2
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_scaled = scaler.fit_transform(X)

chi_scores, p_values = chi2(X_scaled, y)

chi_results = pd.DataFrame({
    "Feature": X.columns,
    "Chi_Square_Score": chi_scores,
    "P_Value": p_values
}).sort_values(
    by="Chi_Square_Score",
    ascending=False
)

print("Chi-Square Feature Scores:")
display(chi_results)

Chi-Square Feature Scores:


,Feature,Chi_Square_Score,P_Value
3,Credit_Score,1.837158,0.175284
1,Income,1.763710,0.184162
2,Experience,1.629682,0.201747
0,Age,1.585586,0.207958
4,Random_Feature,0.021404,0.883683


In [17]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

rfe_model = LogisticRegression(max_iter=2000)

rfe_selector = RFE(
    estimator=rfe_model,
    n_features_to_select=3
)

rfe_selector.fit(X_scaled, y)

rfe_results = pd.DataFrame({
    "Feature": X.columns,
    "Selected": rfe_selector.support_,
    "Ranking": rfe_selector.ranking_
})

print("RFE Results:")
display(rfe_results)

RFE Results:


,Feature,Selected,Ranking
0,Age,True,1
1,Income,True,1
2,Experience,False,2
3,Credit_Score,True,1
4,Random_Feature,False,3


In [21]:
print("Lasso Feature Selection completed successfully!")

display(lasso_results)

Lasso Feature Selection completed successfully!


,Feature,Coefficient,Selected
0,Age,0.000000,False
1,Income,0.000099,True
2,Experience,-0.092717,True
3,Credit_Score,-0.007486,True
4,Random_Feature,-0.005371,True


In [25]:
import pandas as pd
import numpy as np

comparison_results = pd.DataFrame({
    "Feature": X.columns,
    "Correlation": [
        abs(X[column].corr(y))
        for column in X.columns
    ]
})

comparison_results["Selected"] = (
    comparison_results["Correlation"] > 0.1
)

comparison_results = comparison_results.sort_values(
    by="Correlation",
    ascending=False
)

print("Feature Selection Comparison:")
display(comparison_results)

Feature Selection Comparison:


,Feature,Correlation,Selected
0,Age,0.850887,True
1,Income,0.846371,True
3,Credit_Score,0.844151,True
2,Experience,0.787788,True
4,Random_Feature,0.093807,False


In [26]:
from sklearn.feature_selection import chi2
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_scaled = scaler.fit_transform(X)

chi_scores, p_values = chi2(X_scaled, y)

chi_results = pd.DataFrame({
    "Feature": X.columns,
    "Chi_Square_Score": chi_scores,
    "P_Value": p_values
}).sort_values(
    by="Chi_Square_Score",
    ascending=False
)

print("Chi-Square Feature Scores:")
display(chi_results)

Chi-Square Feature Scores:


,Feature,Chi_Square_Score,P_Value
3,Credit_Score,1.837158,0.175284
1,Income,1.763710,0.184162
2,Experience,1.629682,0.201747
0,Age,1.585586,0.207958
4,Random_Feature,0.021404,0.883683


In [27]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

rfe_model = LogisticRegression(max_iter=2000)

rfe_selector = RFE(
    estimator=rfe_model,
    n_features_to_select=3
)

rfe_selector.fit(X_scaled, y)

rfe_results = pd.DataFrame({
    "Feature": X.columns,
    "Selected": rfe_selector.support_,
    "Ranking": rfe_selector.ranking_
})

print("RFE Results:")
display(rfe_results)

RFE Results:


,Feature,Selected,Ranking
0,Age,True,1
1,Income,True,1
2,Experience,False,2
3,Credit_Score,True,1
4,Random_Feature,False,3


In [29]:
from sklearn.ensemble import RandomForestClassifier

tree_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

tree_model.fit(X, y)

tree_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": tree_model.feature_importances_
}).sort_values(
    by="Importance",
    ascending=False
)

print("Tree-Based Feature Importance:")
display(tree_importance)

Tree-Based Feature Importance:


,Feature,Importance
3,Credit_Score,0.27
1,Income,0.25
0,Age,0.24
2,Experience,0.23
4,Random_Feature,0.01


In [30]:
comparison = pd.DataFrame({
    "Method": [
        "Mutual Information",
        "Chi-Square",
        "RFE",
        "L1/Lasso",
        "Tree-Based Importance"
    ],
    "Approach": [
        "Measures statistical dependence",
        "Tests categorical association",
        "Repeatedly removes weak features",
        "Uses L1 regularization",
        "Uses tree model importance"
    ]
})

display(comparison)

,Method,Approach
0,Mutual Information,Measures statistical dependence
1,Chi-Square,Tests categorical association
2,RFE,Repeatedly removes weak features
3,L1/Lasso,Uses L1 regularization
4,Tree-Based Importance,Uses tree model importance


In [31]:
feature_selection_summary = pd.DataFrame({
    "Feature_Selection_Method": [
        "Filter Methods",
        "Wrapper Methods",
        "Embedded Methods",
        "Variance Threshold",
        "Correlation-Based Selection",
        "Mutual Information",
        "Chi-Square",
        "RFE",
        "L1/Lasso",
        "Tree-Based Importance"
    ],
    "Purpose": [
        "Select features using statistical tests",
        "Evaluate feature subsets using models",
        "Select features during model training",
        "Remove low-variance features",
        "Identify redundant features",
        "Measure information shared with target",
        "Measure categorical association",
        "Recursively eliminate weak features",
        "Reduce unnecessary coefficients",
        "Rank features using tree models"
    ]
})

display(feature_selection_summary)

,Feature_Selection_Method,Purpose
0,Filter Methods,Select features using statistical tests
1,Wrapper Methods,Evaluate feature subsets using models
2,Embedded Methods,Select features during model training
3,Variance Threshold,Remove low-variance features
4,Correlation-Based Selection,Identify redundant features
5,Mutual Information,Measure information shared with target
6,Chi-Square,Measure categorical association
7,RFE,Recursively eliminate weak features
8,L1/Lasso,Reduce unnecessary coefficients
9,Tree-Based Importance,Rank features using tree models
